# Tutorial 2 — Transformer Internals

By the end you will be able to:

- run a forward pass by hand and read the **shape** of everything it returns;
- explain what a **hidden state** is, and show that the same word has different ones in different contexts;
- read an **attention matrix**, and verify for yourself that a causal model cannot look forward;
- turn final hidden states into **logits**, then into a next-token distribution;
- write the **generation loop** yourself, and get the same output as `model.generate()`;
- choose a **decoding strategy** — greedy, temperature, top-k, top-p, beam search — and say what each
  one trades away;
- explain what the **KV cache** stores and measure what it saves;
- say where a model's **parameters** actually live.

## 0. Mental model: one forward pass

<img src="images/transformer_cards/00_forward_pass.png" alt="Forward pass card: token ids ride a conveyor through embeddings, transformer blocks and a logits table; shapes go [B,S] to [B,S,H] to [B,S,V]. One pass gives scores; generation repeats." width="650">

A forward pass is a fixed pipeline with no loops and no decisions:

![A forward pass drawn as tensors: input_ids of length 6 become a 6-by-d embedding matrix, pass through N transformer blocks that keep the same shape at every layer, produce 6 final hidden states, and are mapped by a shared LM head weight matrix to a 6-by-vocabulary logit matrix whose last row is used for generation](images/transformer.png)

Three things are worth fixing in your head before any code runs.

**Every stage keeps the sequence dimension.** A 6-token input produces 6 vectors at every layer and 6 rows
of logits at the end. The model scores the next token at *every* position simultaneously, not just the last
one — that is what makes training efficient. Generation simply ignores all rows but the last.

**Depth is refinement, not translation.** Layer 1 does not produce "words" and layer 12 "meaning". Each
block reads the whole sequence and writes an updated vector per position, gradually mixing context into
each one.

**One head, applied everywhere.** The LM head is a single weight matrix `W_vocab`, not one per position.
The same matrix maps every hidden state to vocabulary scores — which is why the logits come out as a
rectangle, and why a bigger vocabulary costs parameters (tutorial 1, section 4) rather than depth.

**Nothing here is generation.** One forward pass produces one distribution over next tokens. Generating
20 tokens means running this 20 times, which is where the KV cache in section 8 earns its keep.

## 1. Install the libraries

<img src="images/transformer_cards/01_install.png" alt="Install card: a toolbox holding transformers, torch and matplotlib, with the pip install command." width="650">

In [ ]:
!pip -q install -U transformers torch matplotlib

## 2. Load a model and read its shape

<img src="images/transformer_cards/02_load_inspect.png" alt="Load and inspect card: GPT-2 as a 12-storey building, with from_pretrained, attn_implementation=eager, model.eval() and config values n_layer=12, n_head=12, n_embd=768, n_positions=1024." width="650">

`AutoModelForCausalLM` loads the weights *plus* the language-modelling head that turns hidden states into
vocabulary scores. `AutoModel` alone would give you the body without that head — useful for embeddings,
useless for generation.

We use GPT-2 (124M parameters) because it runs on a CPU in seconds and its architecture is the plain
decoder every modern LLM still resembles.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_ID = "gpt2"
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, attn_implementation="eager")
model.eval()

cfg = model.config
print("layers:       ", cfg.n_layer)
print("attention heads per layer:", cfg.n_head)
print("hidden size:  ", cfg.n_embd)
print("vocab size:   ", cfg.vocab_size)
print("max context:  ", cfg.n_positions)
print()
total = sum(p.numel() for p in model.parameters())
print(f"total parameters: {total:,}")

### Code walkthrough — config and `eval()`

- **`model.eval()`** turns off dropout, so the same input always gives the same output.
- **The config is the architecture:** `n_layer` blocks, `n_head` attention heads per block, `n_embd` width of
  each position's vector, `n_positions` maximum context. Other models name them `num_hidden_layers`,
  `num_attention_heads`, `hidden_size`.
- **`attn_implementation="eager"`** computes the full `seq × seq` attention matrix that section 4 needs. The
  default fused backends (SDPA / FlashAttention) are faster but never build it. Use eager for inspection only.

## 3. The forward pass

<img src="images/transformer_cards/03_model_outputs.png" alt="Model outputs card: a suitcase holding logits [B,S,V], hidden_states L+1 x [B,S,H] and attentions L x [B,heads,S,S], opened with torch.no_grad(); a side panel shows context changing the vector for &#x27;bank&#x27;." width="650">

By default a model returns only logits. Asking for `output_hidden_states` and `output_attentions` makes it
hand back the intermediate values too, which is the whole point of this notebook.

In [ ]:
text = "The capital of France is"
inputs = tok(text, return_tensors="pt")

with torch.no_grad():
    out = model(**inputs, output_hidden_states=True, output_attentions=True)

n_tokens = inputs["input_ids"].shape[1]
print("tokens:", tok.convert_ids_to_tokens(inputs["input_ids"][0]))
print()
print("input_ids    ", tuple(inputs["input_ids"].shape), "  [batch, seq]")
print("logits       ", tuple(out.logits.shape), "  [batch, seq, vocab]")
print("hidden_states", len(out.hidden_states), "tensors of", tuple(out.hidden_states[0].shape), " [batch, seq, hidden]")
print("attentions   ", len(out.attentions), "tensors of", tuple(out.attentions[0].shape), "[batch, heads, seq, seq]")

### Code walkthrough — reading the shapes

- **`torch.no_grad()`** skips gradient tracking. It isn't needed for inference and costs memory.
- **`hidden_states` has `n_layer + 1` entries.** Index `0` is the embedding output, before any block runs;
  index `i` is the output of block `i`.
- **`attentions` has `n_layer` entries**, one per block, each `[batch, heads, seq, seq]`.
- **`logits` is `[batch, seq, vocab]`**: one vocabulary-sized score vector at *every* position.

### Before you run

The next cell compares each token's vector before any block runs (`hidden_states[0]`) with its vector after
all twelve (`hidden_states[-1]`), using cosine similarity.

The similarities come back low — often near zero, sometimes negative. That is the point: the final vector at
a position is **not** a refined version of that word's embedding, it is a representation of that position
*in this sentence*. The embedding of `" France"` is the same tensor in every text that contains it; its
final hidden state is not.

This is what "contextual representation" means, and it is the single idea separating transformers from the
static word vectors (word2vec, GloVe) that preceded them.

In [ ]:
import torch.nn.functional as F

first = out.hidden_states[0][0]    # after embedding, before any block
last  = out.hidden_states[-1][0]   # after all 12 blocks

print(f"{'token':>10}  {'cos(embedding, final)':>22}  {'‖embedding‖':>12}  {'‖final‖':>10}")
for i, t in enumerate(tok.convert_ids_to_tokens(inputs["input_ids"][0])):
    sim = F.cosine_similarity(first[i], last[i], dim=0).item()
    print(f"{t:>10}  {sim:>22.3f}  {first[i].norm():>12.1f}  {last[i].norm():>10.1f}")

### Code walkthrough — how far the vectors travel

**`F.cosine_similarity`** compares direction and ignores length (`1` same, `0` unrelated, `-1` opposite).
Norms grow with depth because of layer norm and the residual stream, not because meaning changes, so
direction is the right thing to compare.

Try `"The bank raised rates"` against `"The river bank was steep"`: `bank` starts from the same embedding but
ends with a different final hidden state. Every downstream task in this course relies on that.

## 4. Attention: who looks at whom

<img src="images/transformer_cards/04_attention.png" alt="Attention card: robots at desks for &#x27;The cat sat on&#x27; cannot see future tokens behind a no-peeking sign; a lower-triangular 4x4 attention matrix whose rows sum to 1 and future weights are 0." width="650">

Each attention head produces a `seq × seq` matrix. Row `i` is a probability distribution: how much position
`i` draws from every position when building its next representation.

Two properties are worth checking rather than taking on faith, and the next cell checks both.

In [ ]:
import numpy as np

LAYER, HEAD = 5, 4
attn = out.attentions[LAYER][0, HEAD].numpy()

print("shape:", attn.shape)
print("row sums (each row is a distribution):", attn.sum(axis=1).round(3))

upper = attn[np.triu_indices(n_tokens, k=1)]
print("largest weight above the diagonal:", f"{upper.max():.2e}")
print("→ a causal model cannot attend to the future" if upper.max() < 1e-6 else "→ unexpected: future leakage")

### Code walkthrough — two invariants

- **Rows sum to 1.** Each row is a softmax over key positions, so it is position `i`'s attention budget. Rows
  that don't sum to 1 are pre-softmax scores.
- **The upper triangle is zero.** `np.triu_indices(n, k=1)` selects future positions. A causal model masks
  them to `-inf` before the softmax, which lets a decoder train on every position at once without seeing the
  token it is predicting. Encoders like BERT have no mask: they read in both directions but cannot generate.

In [ ]:
import matplotlib.pyplot as plt

labels = [t.replace("\u0120", " ").strip() or "␣" for t in tok.convert_ids_to_tokens(inputs["input_ids"][0])]

fig, ax = plt.subplots(figsize=(5.4, 4.6))
im = ax.imshow(attn, cmap="Blues", vmin=0.0, vmax=1.0)

ax.set_xticks(range(n_tokens), labels, rotation=45, ha="right")
ax.set_yticks(range(n_tokens), labels)
ax.set_xlabel("attended to  (key)")
ax.set_ylabel("attending from  (query)")
ax.set_title(f"Attention weights — layer {LAYER}, head {HEAD}", pad=10)

ax.tick_params(length=0)
for side in ax.spines.values():
    side.set_visible(False)

fig.colorbar(im, ax=ax, shrink=0.82, label="attention weight")
plt.tight_layout()
plt.show()

### Code walkthrough — reading the map

- **`cmap="Blues"`, `vmin=0, vmax=1`:** attention weight is a magnitude, so one hue reads correctly. The fixed
  scale keeps plots of different heads comparable.
- **Lower-triangular shape** is the causal mask.
- **The dark first column is an attention sink.** Because each row must sum to 1, heads put leftover
  attention on token 0. In GPT-2, 126 of 144 heads send more than half their attention there.
- **Heads specialise.** Change `LAYER` / `HEAD`: some heads track the previous token, some the first token,
  some punctuation. Many look like noise, so interpretability is harder than diagrams suggest.

## 5. From hidden states to a next token

<img src="images/transformer_cards/05_next_token.png" alt="Next-token card: only the last logits row goes through a softmax funnel, producing a ranked list of next-token probabilities that sum to 1 (illustrative numbers)." width="650">

The **LM head** is a single linear layer mapping each `n_embd`-wide hidden state to one score per vocabulary
entry. Those raw scores are **logits**; a softmax turns them into probabilities.

In [ ]:
next_logits = out.logits[0, -1]              # last position only
probs = torch.softmax(next_logits, dim=-1)

print(f"prompt: {text!r}\n")
top = torch.topk(probs, 5)
for score, idx in zip(top.values, top.indices):
    print(f"{tok.decode(idx)!r:>12}  p = {score:.3f}")

print()
print("logits row:", tuple(next_logits.shape), "= one score per vocabulary entry")
print("probabilities sum to:", f"{probs.sum():.4f}")

### Code walkthrough — the last row only

- **`out.logits[0, -1]`**: at inference only the last position predicts what comes next. The other rows are
  what give training a target at every position.
- **Softmax is monotonic.** It never changes the argmax. Probabilities matter once you sample, set a
  temperature or use `top_p` (section 7).
- **GPT-2 small does not answer `" Paris"`.** Its top token is `" the"` at about 0.09, and the top 5 are
  nearly flat. A 124M model is not a reliable knowledge store, and a flat distribution is what low confidence
  looks like. A larger checkpoint (e.g. `Qwen/Qwen3-0.6B`) gives a sharper distribution.

## 6. Generation is this cell in a loop

<img src="images/transformer_cards/06_generation_loop.png" alt="Generation loop card: a track with four stops, forward, last row, pick with argmax, append with torch.cat, adding up to 8 new tokens after the unchanged prompt." width="650">

Everything needed to generate text is now on the table: run a forward pass, take the last row, pick a token,
append it, repeat. The next cell does exactly that, then checks the result against `model.generate()`.

In [ ]:
ids = inputs["input_ids"]

for _ in range(8):
    with torch.no_grad():
        logits = model(ids).logits
    next_id = logits[0, -1].argmax().view(1, 1)
    ids = torch.cat([ids, next_id], dim=1)

manual = tok.decode(ids[0])

builtin = tok.decode(
    model.generate(**inputs, max_new_tokens=8, do_sample=False,
                   pad_token_id=tok.eos_token_id)[0]
)

print("by hand:  ", manual)
print("generate():", builtin)
print()
print("identical:", manual == builtin)

### Code walkthrough — the loop, and why it matches

- **`.argmax()`** is greedy decoding. **`.view(1, 1)`** reshapes the token to `[batch, seq]`, and
  **`torch.cat(..., dim=1)`** appends it to the sequence.
- The output matches `generate()` because **`do_sample=False`** with one beam *is* greedy decoding.
- **`pad_token_id=tok.eos_token_id`** silences a warning. GPT-2 has no pad token.
- Each step re-runs the model over the **whole** sequence, recomputing everything from the previous step.
  Section 8 removes that waste.

## 7. Decoding: choosing from the distribution

Section 5 produced a probability distribution; section 6 took its `argmax`. That second step was a
**choice**, and it is a choice you make rather than one the model makes.

This matters more than it first appears. **Decoding is not part of the model.** The weights are fixed, the
forward pass is deterministic, and the distribution is whatever it is — everything below changes only how a
token is drawn from it. The same model produces careful prose or incoherent rambling depending on these
settings alone, which is why "the model is bad" is so often a decoding problem instead.

In [ ]:
common = dict(pad_token_id=tok.eos_token_id)

greedy_long = model.generate(**inputs, do_sample=False, max_new_tokens=40, **common)
print(tok.decode(greedy_long[0]))

### Code walkthrough — why greedy collapses

On open-ended text, greedy decoding falls into a loop. Each repeated phrase becomes evidence for repeating it
again, and a deterministic choice can't break the cycle. The text stays fluent; it is just bland, because the
*most probable* sequence is not the same as *good* text. The model's probabilities are reasonable here. The
problem is how we pick from them.

### 7.1 Temperature: reshaping before you sample

<img src="images/transformer_cards/07_1_temperature.png" alt="Temperature card: softmax(logits / T); T=0.5 gives a sharper distribution, T=1.0 the original, T=1.5 a flatter one; the ranking stays the same, only the gaps change." width="650">

**Temperature** divides the logits before the softmax. Below 1 it sharpens the distribution toward the
leader; above 1 it flattens it toward uniform.

In [ ]:
with torch.no_grad():
    last_logits = model(**inputs).logits[0, -1]

print(f"prompt: {text!r}\n")
for T in (0.5, 1.0, 1.5):
    probs_T = torch.softmax(last_logits / T, dim=-1)
    top = torch.topk(probs_T, 4)
    row = "   ".join(f"{tok.decode(i).strip()!r}: {v:.3f}" for v, i in zip(top.values, top.indices))
    print(f"T = {T:<4}  {row}")

### Code walkthrough — what the numbers show

The top token goes from ~0.40 (`T=0.5`) to ~0.09 (`T=1.0`) to ~0.02 (`T=1.5`). The ranking never changes; only
the gaps do, and sampling responds to the gaps.

- **`T < 1`** concentrates probability on the top tokens. As `T → 0` it becomes greedy.
- **`T = 1`** keeps the model's own distribution.
- **`T > 1`** raises the tail: more variety, less coherence.

Temperature has the biggest effect on flat, uncertain distributions like this one and very little on
confident ones.

### 7.2 Truncation: top-k and top-p

<img src="images/transformer_cards/07_2_top_k_top_p.png" alt="Top-k vs top-p card: a top-k bouncer admits a fixed 50 candidates, while a top-p bouncer keeps adding the most likely tokens until their cumulative probability reaches 0.9." width="650">

A flat distribution has a very long tail. Sampling from all 50,257 tokens means that any individual absurd
token is unlikely, but *some* absurd token is nearly certain over a few dozen steps.

Both fixes cut the tail before sampling. **top-k** keeps a fixed number of candidates; **top-p** (nucleus
sampling) keeps however many are needed to reach a cumulative probability, so the candidate set shrinks when
the model is confident and grows when it is not.

In [ ]:
from transformers import set_seed

strategies = {
    "greedy":                  dict(do_sample=False),
    "pure sampling (T=1)":     dict(do_sample=True, temperature=1.0, top_k=0, top_p=1.0),
    "top-k = 50":              dict(do_sample=True, temperature=1.0, top_k=50),
    "top-p = 0.9, T = 0.8":    dict(do_sample=True, temperature=0.8, top_p=0.9),
}

for name, settings in strategies.items():
    set_seed(0)
    sampled = model.generate(**inputs, max_new_tokens=25, **settings, **common)
    print(f"--- {name}")
    print(tok.decode(sampled[0]), "\n")

### Code walkthrough — reading the four outputs

- **`set_seed(0)`** makes sampling reproducible so the strategies can be compared. Greedy has no randomness,
  so it is unaffected.
- **Pure sampling** (`top_k=0, top_p=1.0`) samples from the whole long tail and produces fluent nonsense.
- **`top_k=50`** is a fixed cutoff: too many candidates when the model is sure, sometimes too few when it
  isn't.
- **`top_p=0.9`** keeps the smallest set of tokens whose probabilities add up to 0.9, so it adapts to the
  model's confidence. It is the standard choice for open-ended generation.
- **`top_p ≈ 0.9` with `T` slightly below 1** is the practical default. It stops the degeneration but does
  nothing for accuracy: no sampling setting makes a 124M model know facts.

### 7.3 Beam search

<img src="images/transformer_cards/07_3_beam_search.png" alt="Beam search card: from &#x27;A dog is&#x27;, four paths are explored at once with their scores, and the best-scoring hypothesis wins; more beams cost more compute and memory." width="650">

Greedy commits to the best token at each step. **Beam search** keeps `num_beams` partial sequences alive and
chooses the one with the best total probability at the end — a better search for the high-probability
sequence.

In [ ]:
beam = model.generate(**inputs, num_beams=4, do_sample=False,
                      early_stopping=True, max_new_tokens=25, **common)
print(tok.decode(beam[0]))

### Code walkthrough — a better search for the wrong thing

Beam search finds higher-probability sequences than greedy and still loops. The high-probability sequences
really are repetitive, so a more thorough search finds more repetition.

- **Use it for closed-ended tasks** (translation, summarisation), with `num_beams=4` or `5`.
- **Avoid it for open-ended generation**, where you want interesting text, not the most likely text.

Compute and memory grow linearly with the number of beams. **`early_stopping=True`** stops once all beams have
finished.

### 7.4 Repetition controls

<img src="images/transformer_cards/07_4_repetition.png" alt="Repetition card: a &#x27;ha ha ha&#x27; echo tunnel, with repetition_penalty=1.2 as a speed bump that discourages seen tokens and no_repeat_ngram_size=3 as a barrier that bans repeated 3-token phrases." width="650">

If a loop persists, two arguments attack it directly.

In [ ]:
fixed = model.generate(**inputs, do_sample=False, max_new_tokens=40,
                       repetition_penalty=1.2, no_repeat_ngram_size=3, **common)
print(tok.decode(fixed[0]))

### Code walkthrough — two blunt instruments

- **`repetition_penalty=1.2`** lowers the scores of tokens already in the context. `1.0` turns it off;
  `1.1`–`1.3` is the usable range. Above that, the model starts avoiding words it needs.
- **`no_repeat_ngram_size=3`** bans any 3-token sequence from appearing twice. The ban is absolute, which is
  risky for names, technical terms and code.

The loop goes away, but the text is no more accurate. For real applications, a larger or instruction-tuned
model fixes repetition better than stronger penalties.

### 7.5 When generation stops

<img src="images/transformer_cards/07_5_stopping.png" alt="Stopping card: a token train meets three signals, max_new_tokens=40 as a budget, eos_token_id as a finish flag, and stop_strings=[&#x27;.&#x27;] as a text pattern." width="650">

Three mechanisms, and a model that "won't stop" is nearly always missing one.

In [ ]:
unbounded = model.generate(**inputs, do_sample=False, max_new_tokens=40, **common)
stopped   = model.generate(**inputs, do_sample=False, max_new_tokens=40,
                           stop_strings=["."], tokenizer=tok, **common)

print("max_new_tokens only:\n", tok.decode(unbounded[0]), "\n")
print("stop_strings=['.']:\n", tok.decode(stopped[0]), "\n")
print("eos token:", repr(tok.eos_token), "| id:", tok.eos_token_id)

### Code walkthrough — the three stopping mechanisms

- **`max_new_tokens`** is the hard limit and the only stop that always fires. Always set it.
- **EOS token:** base models like GPT-2 rarely emit it; instruction-tuned models do. If a chat model never
  stops, check the chat template.
- **`stop_strings`** (pass `tokenizer=tok` too) stops at a text pattern such as `"\n\n"` or a role marker.
  The tokens are generated first and cut afterwards, so you still pay for them.

### 7.6 Streaming

<img src="images/transformer_cards/07_6_streaming.png" alt="Streaming card: tokens are handed to a reader one by one as they are produced, using TextStreamer(tok, skip_prompt=True); earlier display, not faster computation." width="650">

Generation is sequential, so tokens exist one at a time. Waiting for all of them before showing anything is a
choice, and usually the wrong one for an interface.

In [ ]:
from transformers import TextStreamer

_ = model.generate(**inputs, do_sample=False, max_new_tokens=20,
                   streamer=TextStreamer(tok, skip_prompt=True), **common)

### Code walkthrough — streaming changes nothing but the wait

**`TextStreamer(tok, skip_prompt=True)`** prints each token as it is generated, without repeating the prompt.
Total time stays the same; what improves is time-to-first-token, the delay the user actually notices. In an
application, use **`TextIteratorStreamer`** and run `generate` on a background thread so tokens can be
forwarded as they arrive.

## 8. The KV cache

<img src="images/transformer_cards/08_kv_cache.png" alt="KV cache card: keys and values from past tokens are stored in a fridge and reused, so only the new token&#x27;s K and V are computed; faster decoding at the cost of more memory." width="650">

Attention at position `i` needs a key and a value vector for every position `≤ i`. In the loop above, those
were recomputed from scratch on every iteration — for a 500-token generation, position 1's key gets computed
500 times.

The **KV cache** keeps them. Each step then computes keys and values for the *one* new token and reads the
rest from memory, turning generation from quadratic in total work to linear.

In [ ]:
import time

def time_generation(use_cache, new_tokens=60):
    t0 = time.perf_counter()
    model.generate(**inputs, max_new_tokens=new_tokens, do_sample=False,
                   use_cache=use_cache, pad_token_id=tok.eos_token_id)
    return time.perf_counter() - t0

with_cache = time_generation(True)
without    = time_generation(False)

print(f"use_cache=True :  {with_cache:.2f}s")
print(f"use_cache=False:  {without:.2f}s")
print(f"speed-up:         {without / with_cache:.1f}×")

### Code walkthrough — what you just measured

- **The text is identical either way.** The cache only avoids recomputation.
- **The speed-up grows with length:** the work saved grows quadratically, the work kept linearly. Try
  `new_tokens=200`.
- **The cost is memory**, roughly `2 × layers × heads × head_dim × seq_len × batch × bytes_per_value`. That is
  why long-context serving is limited by memory, and why multi-query / grouped-query attention and paged
  attention exist.
- `use_cache=True` is the default.

## 9. Where the parameters are

<img src="images/transformer_cards/09_parameters.png" alt="Parameters card: jars for token embeddings (vocab_size x n_embd), position embeddings (n_positions x n_embd) and transformer blocks, with bytes per value for FP32, FP16/BF16 and INT8." width="650">

"124M parameters" is not one undifferentiated blob. Knowing its split explains several practical things at
once — why vocabulary size is expensive, and why fine-tuning strategies target the blocks.

In [ ]:
emb_tokens = model.transformer.wte.weight.numel()
emb_pos    = model.transformer.wpe.weight.numel()
embeddings = emb_tokens + emb_pos
blocks     = total - embeddings

print(f"token embeddings     {emb_tokens:>12,}   ({emb_tokens/total:.1%})")
print(f"position embeddings  {emb_pos:>12,}   ({emb_pos/total:.1%})")
print(f"transformer blocks   {blocks:>12,}   ({blocks/total:.1%})")
print(f"{'':21}{'-'*12}")
print(f"total                {total:>12,}")
print()
print(f"per block            {blocks // cfg.n_layer:>12,}")
print(f"weights in fp32      {total * 4 / 1e6:>12.0f} MB")

### Code walkthrough — reading the budget

- **`wte`** (`vocab_size × n_embd`) and **`wpe`** (`n_positions × n_embd`) are lookup tables, not
  computation.
- **Token embeddings are about a third of GPT-2 small.** This is the cost of vocabulary size from tutorial 1:
  each extra token adds a row of `n_embd` parameters. In a 7B model the same table is a tiny share, so large
  models can afford bigger vocabularies.
- **`total * 4`** bytes is the size in fp32. Halve it for fp16/bf16 and quarter it for int8. To check whether
  a model fits on a GPU, add the KV cache from section 8.

## 10. Three things that will bite you

<img src="images/transformer_cards/10_inference_traps.png" alt="Inference traps card: model.eval() and torch.no_grad() as two separate switches, attention_mask shielding against pad tokens, and the tokenizer and model as matching puzzle pieces from the same checkpoint." width="650">

**`eval()` and `no_grad()` are different switches.** `eval()` changes layer *behaviour* (dropout, batchnorm);
`no_grad()` stops gradient *tracking*. Inference wants both. Neither is on by default.

**Padding without a mask corrupts results.** Batch two sentences of different lengths and the short one is
padded — then attention happily attends to the pad tokens. `tokenizer(...)` returns an `attention_mask`
precisely so this does not happen, and passing `**inputs` forwards it. Build `input_ids` by hand and you
must pass the mask by hand too.

**A tokenizer/model mismatch stays silent here as well.** Tutorial 1 made this point for decoding; it holds
for embeddings. Id 2000 indexes row 2000 of `wte` whatever tokenizer produced it, so the wrong pairing gives
you fluent nonsense rather than an error.